<a href="https://colab.research.google.com/github/maverick0721/Fine-Tuning-SLM/blob/main/FineTuning_SLM.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [34]:
!pip install -U torch torchvision torchaudio xformers --index-url https://download.pytorch.org/whl/cu128
!pip install -U unsloth
!pip install -U transformers==4.56.2 datasets==4.3.0
!pip install -U --no-deps trl==0.22.2

import torch
assert torch.cuda.is_available(), "Enable GPU runtime (Colab → Runtime → GPU)"

Looking in indexes: https://download.pytorch.org/whl/cu128


In [35]:
from dataclasses import dataclass
from typing import Optional, List
from datasets import load_dataset
from unsloth import FastLanguageModel
from peft import PeftModel
from trl import SFTTrainer, SFTConfig

In [36]:
@dataclass
class FineTuneConfig:
    # model
    model_name: str
    load_in_4bit: bool = True
    max_seq_length: int = 4096
    dtype: Optional[str] = None  # None = auto

    # dataset
    dataset_name: str = "tatsu-lab/alpaca"
    split: str = "train"
    seed: int = 3407

    # training
    output_dir: str = "outputs"
    lora_save_path: str = "lora_adapters"
    per_device_bs: int = 2
    grad_acc_steps: int = 4
    epochs: int = 1
    lr: float = 2e-5
    warmup_ratio: float = 0.1
    logging_steps: int = 10
    packing: bool = True

    # lora
    lora_r: int = 32
    lora_alpha: int = 32
    lora_dropout: float = 0.0
    target_modules: List[str] = None
    use_gc: bool = False

    # save merged model
    save_merged: bool = False
    merged_save_path: str = "merged_fp16_model"

In [28]:
%mv /content/FineTuning-SLM.ipynb /content/Fine-Tuning-SLM

In [32]:
!git add --all

In [33]:
!git commit -m "Added/Moved Notebook sucessfully to Github Repo"

[main 721d85c] Added/Moved Notebook sucessfully to Github Repo
 1 file changed, 1 insertion(+)
 create mode 100644 FineTuning-SLM.ipynb


In [37]:
class UnslothFineTuner:
    def __init__(self, cfg: FineTuneConfig):
        self.cfg = cfg
        if self.cfg.target_modules is None:
            self.cfg.target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]
        self.model = None
        self.tokenizer = None

    #Load Model
    def load_model(self):
        print(f"✅ Loading model: {self.cfg.model_name}")

        self.model, self.tokenizer = FastLanguageModel.from_pretrained(
            model_name=self.cfg.model_name,
            max_seq_length=self.cfg.max_seq_length,
            dtype=self.cfg.dtype,
            load_in_4bit=self.cfg.load_in_4bit,
        )

        self.model = FastLanguageModel.get_peft_model(
            self.model,
            r=self.cfg.lora_r,
            target_modules=self.cfg.target_modules,
            lora_alpha=self.cfg.lora_alpha,
            lora_dropout=self.cfg.lora_dropout,
            bias="none",
            use_gradient_checkpointing=self.cfg.use_gc,
            random_state=self.cfg.seed,
        )

        self.model.print_trainable_parameters()
        print("Is PEFT model?", isinstance(self.model, PeftModel))
        print("Device:", next(self.model.parameters()).device, " | Dtype:", next(self.model.parameters()).dtype)


In [70]:
# Copy the currently running notebook to the repo folder
!cp /content/FineTuning-SLM.ipynb ./FineTuning-SLM.ipynb

cp: cannot stat '/content/FineTuning-SLM.ipynb': No such file or directory


In [64]:
%cd /content/Fine-Tuning-SLM

/content/Fine-Tuning-SLM


In [67]:
!git add .

In [68]:
!git commit -m "Loading Model"

On branch main
Your branch is up to date with 'origin/main'.

nothing to commit, working tree clean
